## RNN (Recurrant Neural Network) Basics

In [1]:
from keras import Sequential, Input
from keras.layers import Dense, SimpleRNN

In [2]:
model = Sequential([
    Input(shape = (4, 5))
])
model.add(SimpleRNN(3, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 3)              │            27 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31 (124.00 B)

 Trainable params: 31 (124.00 B)

 Non-trainable params: 0 (0.00 B)

In [4]:
print(model.get_weights()[0].shape)
model.get_weights()[0]

(5, 3)


array([[-0.732183  , -0.2393511 ,  0.61872   ],
       [ 0.37052113, -0.76166886, -0.78867286],
       [-0.8619037 ,  0.6809419 ,  0.5835772 ],
       [ 0.7746133 ,  0.4522528 ,  0.39371425],
       [ 0.2600382 , -0.18702829,  0.78280336]], dtype=float32)

In [5]:
print(model.get_weights()[1].shape)
model.get_weights()[1]

(3, 3)


array([[ 0.4402213 , -0.2797469 , -0.85319805],
       [ 0.86540633,  0.38547   ,  0.3201325 ],
       [ 0.23932615, -0.8792922 ,  0.41178674]], dtype=float32)

In [6]:
print(model.get_weights()[2].shape)
model.get_weights()[2]

(3,)


array([0., 0., 0.], dtype=float32)

In [7]:
print(model.get_weights()[3].shape)
model.get_weights()[3]

(3, 1)


array([[1.0312763 ],
       [0.754776  ],
       [0.72229946]], dtype=float32)

In [8]:
print(model.get_weights()[4].shape)
model.get_weights()[4]

(1,)


array([0.], dtype=float32)

### Basic Training RNN

#### 1. Data Preparation method

In [10]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
from keras.utils import pad_sequences
from keras.datasets import imdb
from keras import Sequential, Input
from keras.layers import Dense, SimpleRNN, Embedding, Flatten

In [2]:
docs = [
    "go india", 
    "india india", 
    "hip hip hurray",
    "jeetega bhai jeetega india jeetega",
    "bharat mata ki jai",
    "kohli kohli",
    "sachin sachin",
    "dhoni dhoni",
    "inqlaab zindabad"
]

In [3]:
tokenizer = Tokenizer(oov_token='<nothing>')

In [4]:
tokenizer.fit_on_texts(docs)

In [5]:
tokenizer.word_index

{'<nothing>': 1,
 'india': 2,
 'jeetega': 3,
 'hip': 4,
 'kohli': 5,
 'sachin': 6,
 'dhoni': 7,
 'go': 8,
 'hurray': 9,
 'bhai': 10,
 'bharat': 11,
 'mata': 12,
 'ki': 13,
 'jai': 14,
 'inqlaab': 15,
 'zindabad': 16}

In [6]:
sequences = tokenizer.texts_to_sequences(docs)
sequences

[[8, 2],
 [2, 2],
 [4, 4, 9],
 [3, 10, 3, 2, 3],
 [11, 12, 13, 14],
 [5, 5],
 [6, 6],
 [7, 7],
 [15, 16]]

In [9]:
sequences = pad_sequences(sequences=sequences, padding='post')
sequences

array([[ 8,  2,  0,  0,  0],
       [ 2,  2,  0,  0,  0],
       [ 4,  4,  9,  0,  0],
       [ 3, 10,  3,  2,  3],
       [11, 12, 13, 14,  0],
       [ 5,  5,  0,  0,  0],
       [ 6,  6,  0,  0,  0],
       [ 7,  7,  0,  0,  0],
       [15, 16,  0,  0,  0]], dtype=int32)

### 2. Train an RNN on text data  
The keras imdb dataset is already tokenized into numbers by following the vocabulary index method described above. 

In [12]:
# lets do it with actual data
(X_train, y_train), (X_test, y_test) = imdb.load_data()


In [14]:
print(X_train.shape)
print(X_test.shape)

(25000,)
(25000,)


In [15]:
print(X_train[0])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 22665, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 21631, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 19193, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 10311, 8, 4, 107, 117, 5952, 15, 256, 4, 31050, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 12118, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


In [16]:
## padding
X_train = pad_sequences(X_train, padding='post', maxlen=100)
X_test = pad_sequences(X_test, padding='post', maxlen=100)

In [17]:
model = Sequential([Input(shape=(100, 1))])
model.add(SimpleRNN(32, activation='relu', return_sequences=False))
model.add(Dense(1, activation='sigmoid'))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,121 (4.38 KB)

 Trainable params: 1,121 (4.38 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [19]:
model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.4974 - loss: 33.8966 - val_accuracy: 0.5003 - val_loss: 15.7712
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.5109 - loss: 12.6381 - val_accuracy: 0.5149 - val_loss: 13.7539
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.5105 - loss: 10.9821 - val_accuracy: 0.5046 - val_loss: 8.3974
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.5070 - loss: 7.5975 - val_accuracy: 0.5157 - val_loss: 7.4688
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.5108 - loss: 8.4133 - val_accuracy: 0.5106 - val_loss: 3.2873


### 3. Using Embedding to train the RNN instead of vocabulary word indices

Advantages of using Embedding instead of word index method:

1. **Dense representation** — Word indices are just arbitrary integers (e.g., india=2, jeetega=3). An Embedding maps each word to a dense vector (e.g., 32 or 64 floats), capturing far more information per word.

2. **Semantic similarity** — Similar words end up with similar vectors after training. Word index gives no such relationship — india=2 and bharat=11 look completely unrelated numerically.

3. **Trainable** — Embedding weights are learned during backpropagation, so the model learns representations optimized for the task.

4. **No false ordinal relationship** — Raw indices imply dhoni(7) > india(2), which is meaningless. Embeddings remove this false numerical ordering.

5. **More efficient than one-hot** — One-hot vectors are size = vocab_size (sparse, huge). Embeddings are compact fixed-size vectors regardless of vocab size.

6. **Better generalization** — Because semantically related words have similar vectors, the model generalizes better to unseen combinations.

In [10]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
from keras.utils import pad_sequences
from keras.datasets import imdb
from keras import Sequential, Input
from keras.layers import Dense, SimpleRNN, Embedding, Flatten
import warnings
import numpy as np
warnings.filterwarnings("ignore", category=DeprecationWarning, module="numpy")


In [11]:
(X_train, y_train), (X_test, y_test) = imdb.load_data()

c:\Users\mrinm\Downloads\MyTraining\DeepLearning\CampusX_Deeplearning\.venv\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [12]:
X_train = pad_sequences(X_train, padding='post', maxlen=100)
X_test = pad_sequences(X_test, padding='post', maxlen=100)

In [16]:
print(X_train.shape)
print(X_test.shape)

(25000, 100)
(25000, 100)


In [18]:
vocab_dim = max(max(seq) for seq in X_train) + 1
embedding_dim = 2

In [22]:
model = Sequential([Input(shape=(100, ))])
model.add(Embedding(input_dim=vocab_dim, output_dim=embedding_dim))
model.add(SimpleRNN(32, activation='relu', return_sequences=False))
model.add(Dense(1, activation='sigmoid'))
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 100, 2)         │       177,174 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 32)             │         1,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 178,327 (696.59 KB)

 Trainable params: 178,327 (696.59 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam'
)

model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 17ms/step - loss: 0.6477 - val_loss: 0.4917
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - loss: 0.4724 - val_loss: 0.4968
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 0.3333 - val_loss: 0.4053
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - loss: 0.2255 - val_loss: 0.3882
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - loss: 0.1649 - val_loss: 0.4544
